In [12]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

In [13]:
# .env 파일에서 환경 변수 로드
load_dotenv("../env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = ChatOpenAI(model="gpt-4o-mini")

In [14]:
llm.invoke("안녕")

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 9, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_36b9eb89d1', 'id': 'chatcmpl-EdPx8H4lTeCSdjPQRYzlyVKzmIyUt', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17ae6-1247-74f7-a990-950487e37518-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 9, 'output_tokens': 10, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
# <오픈AI API를 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
import openai
from typing import List

# 기본 오픈AI 클라이언트 사용
client = openai.OpenAI()

# "안녀하세요!" 메시지를 보내고 응답을 받음
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "안녕하세요!"}]
)
response.choices[0].message.content

'안녕하세요! 어떻게 도와드릴까요?'

In [16]:
# 요청에 사용할 프롬프트 템플릿 정의
prompt_template = "주제 {topic}에 대해 짧은 설명을 해주세요"

# 메시지를 보내고 모델의 응답을 받는 함수
def call_chat_model(messages: List[dict]) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    return response.choices[0].message.content

# 주어진 주제에 따라 설명을 요청하는 함수
def invoke_chain(topic: str) -> str:
    prompt_value = prompt_template.format(topic=topic)
    messages = [{"role": "user", "content": prompt_value}]
    return call_chat_model(messages)

# "더블딥" 주제로 설명 요청
invoke_chain("Nvidia Isaac Sim")

'Nvidia Isaac Sim은 로봇 시뮬레이션을 위한 강력한 플랫폼으로, 로봇 기술 개발자와 연구자들이 가상 환경에서 로봇 알고리즘을 테스트하고 검증할 수 있도록 돕는 도구입니다. 이 시뮬레이터는 사실적인 물리 엔진과 고급 그래픽을 기반으로 하여 다양한 로봇 시나리오를 구현할 수 있도록 설계되었으며, 인공지능(AI) 및 머신러닝과의 통합 또한 지원합니다. 이를 통해 사용자들은 실제 환경에서의 리스크를 최소화하면서 동작 계획, 네비게이션, 센서 데이터 처리 등을 실험할 수 있습니다. Isaac Sim은 로봇 연구 및 개발을 가속화하고, 실제 로봇 시스템을 실제 세계에 배포하기 전에 시뮬레이션 환경에서 충분히 테스트할 수 있도록 합니다.'

In [17]:
# <랭체인을 사용하여 주제에 대한 간단한 설명 생성 파이프라인 구축>

# 라이브러리 불러오기
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
# 미스트랄AI 모델을 사용할 경우 주석 해제
# from langchain_mistralai.chat_models import ChatMistralAI

# 주어진 주제에 대해 짧은 설명을 요청하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_template(
    "주제 {topic}에 대해 짧은 설명을 해주세요"
)

# 출력 파서를 문자열로 설정
output_parser = StrOutputParser()
# 오픈AI의 gpt-4o 모델을 사용하여 채팅 모델 설정
model = ChatOpenAI(model="gpt-4o")
# 미스트랄AI 모델을 사용할 경우 주석 해제
# model = ChatMistralAI()

# 파이프라인 설정: 주제를 받아 프롬프트를 생성하고, 모델을 통해 응답을 생성한 후 문자열로 파싱
chain = (
    {"topic": RunnablePassthrough()}  # 입력 받은 주제를 그대로 통과시킴
    | prompt                         # 프롬프트 템플릿을 적용
    | model                          # 모델을 사용해 응답 생성
    | output_parser                  # 응답을 문자열로 파싱
)

# "더블딥" 주제로 설명 요청
chain.invoke("Nvidia Isaac Sim")

'NVIDIA Isaac Sim은 로보틱스 개발을 위한 고급 시뮬레이션 플랫폼입니다. 로봇의 설계, 테스트 및 훈련을 가상 환경에서 수행할 수 있게 도와줍니다. 이 플랫폼은 AI 기반 로봇 시스템의 개발을 가속화하며, 실제 물리 환경을 정밀하게 모사하여 로봇의 센서 데이터 처리, 장애물 회피, 경로 계획 등의 기능을 테스트할 수 있습니다. NVIDIA의 강력한 그래픽 처리 능력을 바탕으로 고품질의 시각적 시뮬레이션을 제공하며, Omniverse를 통해 협업 및 통합이 가능합니다. Isaac Sim은 연구 기관, 교육 기관, 산업체 등에서 로봇의 설계 및 검증에 폭넓게 활용되고 있습니다.'